# Benchmarks - Cluster Initialisation

**Main considerations when implementing clustering initialisation**

* MagmaClust needs an initial mixture
* It needs to be both informative and cheap to compute
* The main characteristic of task that is relevent for clustering is the order of magnitude of the values of this task
* When using distinct HPs, statistics about the variance of each task might be useful
* It must work even on sequences that are not aligned, implying the use of a dimensionality reduction technique

The approach used in the original R implementation is to use k-means on features `[min(t), mean(t), max(t)]` for each sequence t.

We adopt the same strategy, but we also use `[var(t)]` when `distinct_hp == True`.

We should also explore whether normalizing each feature gives better results.

---
## Setup

In [1]:
# Jax configuration
USE_JIT = False
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [2]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from functools import partial

In [3]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)

In [4]:
# Third party
from jax import jit, vmap
from jax.tree_util import register_pytree_node_class
from jax import numpy as jnp
from jax import lax
from jax.lax import cond

import numpy as np
import pandas as pd

In [5]:
# Local
from Kernax import RBFKernel, AbstractKernel, SEMagmaKernel, DiagKernel, ExpKernel
from MagmaClustPy.utils import preprocess_db
from MagmaClustPy.linalg import map_to_full_matrix_batch, map_to_full_array_batch, compute_mapping, lexicographic_sort
from MagmaClustPy.hyperpost import hyperpost

from MagmaClustPy.kmeans import k_means

INFO:2025-11-17 11:26:29,713:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
2025-11-17 11:26:29,713 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/MagmaClustPy/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [6]:
# Config
key = jax.random.PRNGKey(0)
test_db_size = "medium"
test_db_nb_clust = 3

---
## Data

---
## Current implementation

In [7]:
# No implementation yet

---
## Custom implementation(s)

In [8]:
@partial(jit, static_argnums=(1, 2))
def k_means_init(padded_outputs, k, distinct_hp=False):
	"""
	Compute the initial assignment of outputs between k clusters using k_means and a naive dimensionality reduction based on task statistics (min, mean, max)

	:param padded_outputs: the outputs from each task, (jnp.array, shape=(T, Max_N))
	:param k: the number of clusters (int)
	:param distinct_hp: whether distinct hyperparameters are used (bool, default=False)
	:return: the initial mixture as a one-hot encoded array (jnp.array, shape=(T, k))
	"""
	# Compute statistics
	features = jnp.stack([
		jnp.nanmin(padded_outputs, axis=0),  # Min
		jnp.nanmean(padded_outputs, axis=0),  # Mean
		jnp.nanmax(padded_outputs, axis=0)  # Max
	], axis=-1).squeeze()

	if distinct_hp:
		features = jnp.concatenate([
			features,
			jnp.nanvar(padded_outputs, axis=0, ddof=1).reshape(-1, 1)  # Variance
		], axis=-1)

	# Run k-means
	_, labels, _= k_means(features, n_clusters=k, n_init=10, max_iter=100)

	# One-hot encode labels
	labels = jnp.eye(k)[labels]

	return labels

---
## Comparison

### shared Input, shared HP

In [9]:
db = pd.read_csv(f"../datasets/K={test_db_nb_clust}/{test_db_size}_shared_input_shared_hp.csv")
padded_inputs, padded_outputs, mappings, all_inputs = preprocess_db(db)
all_inputs.shape, padded_inputs.shape

((150, 1), (200, 150, 1))

In [10]:
labels = k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False)
labels

Array([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=int32)

In [11]:
%timeit k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False).block_until_ready()

601 ms ± 40.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### shared Input, distinct HP

In [12]:
db = pd.read_csv(f"../datasets/K={test_db_nb_clust}/{test_db_size}_shared_input_distinct_hp.csv")
padded_inputs, padded_outputs, mappings, all_inputs = preprocess_db(db)
all_inputs.shape, padded_inputs.shape

((150, 1), (200, 150, 1))

In [13]:
labels = k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=True)
labels

Array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [14]:
%timeit k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=True).block_until_ready()

609 ms ± 20.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### distinct Input, shared HP

In [15]:
db = pd.read_csv(f"../datasets/K={test_db_nb_clust}/{test_db_size}_distinct_input_shared_hp.csv")
padded_inputs, padded_outputs, mappings, all_inputs = preprocess_db(db)
all_inputs.shape, padded_inputs.shape

((401, 1), (200, 200, 1))

In [16]:
labels = k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False)
labels

Array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2], dtype=int32)

In [17]:
%timeit k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False).block_until_ready()

595 ms ± 18.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### distinct Input, distinct HP

In [18]:
db = pd.read_csv(f"../datasets/K={test_db_nb_clust}/{test_db_size}_distinct_input_distinct_hp.csv")
padded_inputs, padded_outputs, mappings, all_inputs = preprocess_db(db)
all_inputs.shape, padded_inputs.shape

((401, 1), (200, 200, 1))

In [19]:
labels = k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False)
labels

Array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0], dtype=int32)

In [20]:
%timeit k_means_init(padded_outputs, k=test_db_nb_clust, distinct_hp=False).block_until_ready()

655 ms ± 38.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


---
## Conclusion

The Jax implementation of k-means from [this repository](https://github.com/ethqnol/jax-kmeans/tree/main) seems like a good first fit to our needs.

---